# 实验报告 #2

> 姓名：林佳胜      学号：PB24511997
>
> 时间：2026 年 3 月 11 日

---

## 一. 运行环境

- 操作系统：`Windows 10`
- 编程语言：`Python 3.11.9`
- 主要依赖库：`numpy>=2.4.2`

---

## 二. 实验目的

1.  **算法实现**：编写并实现 Jacobi 与 Gauss-Seidel 迭代法求解线性方程组。
2.  **收敛性分析**：对比两种方法在不同阶数 $n$ 下的收敛速度与稳定性。
3.  **病态系统探究**：通过 Hilbert 偏移矩阵测试，理解矩阵性质对迭代收敛的影响。

---


## 三. 实验内容与结果

### 1. 实验内容

本实验研究迭代法求解线性方程组的数值表现，重点比较 **Jacobi** 与 **Gauss-Seidel** 两种方法在不同规模下的收敛性与稳定性。

#### 1.1 问题模型

设
$$
A = H + 1.8I,
$$
其中 $I$ 为单位矩阵，$H$ 为 $n$ 阶 Hilbert 矩阵：
$$
H=(h_{ij})_{n\times n},\qquad h_{ij}=\frac{1}{i+j-1},\quad i,j=1,2,\dots,n.
$$

要求解线性方程组
$$
Ax=b.
$$

#### 1.2 精确解构造

为便于误差评估，先指定精确解为
$$
x^*=(1,1,\dots,1)^T,
$$
再由
$$
b=Ax^*
$$
生成右端向量，从而得到“精确解已知”的测试问题。

#### 1.3 实验设置

1. 迭代方法：Jacobi、Gauss-Seidel。  
2. 初始向量：$x^{(0)}=0$。  
3. 矩阵阶数：$n=10,20,60,100,200,500,2000$。  
4. 停止条件：$\|x^{(k)}-x^*\|_1<10^{-5}$；若迭代步数超过 $5\times 10^5$，视为求解失败。  

#### 1.4 输出指标

对每个 $n$ 和每种方法，记录以下结果：

1. 是否收敛；  
2. 迭代步数 $k$；  
3. 误差 $\|x^{(k)}-x^*\|_1$；  
4. 数值解范数 $\|x^{(k)}\|_1$（保留6位小数，失败时可不报告）。


In [1]:
# 依赖库

from dataclasses import dataclass
from typing import Optional, Callable
import numpy as np


In [2]:
# 数据结构

@dataclass(frozen=True)
class IterConfig:
    tol: float = 1e-5 # 容许量
    max_iter: int = 500000 # 最大迭代步数
    shift: float = 1.8 # 矩阵平移参数
    dtype: type = np.float64 # 数据精度类型

@dataclass
class IterResult:
    n: int
    method: str
    converged: bool # 是否收敛
    iterations: int # 迭代步数
    error_l1: float
    x_l1: Optional[float] # 失败时为 None
    

In [3]:
# 问题模型构造

def hilbert_matrix(
    n: int, 
    dtype: type[np.floating] = np.float64
) -> np.ndarray:
    """构造 Hilbert 矩阵。"""
    i = np.arange(1, n + 1, dtype=dtype)

    # H_ij = 1 / (i + j - 1)
    return 1.0 / (i[:, None] + i[None, :] - 1.0) # 广播机制

def build_problem(
    n: int, 
    cfg: IterConfig
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """问题建构。"""
    h = hilbert_matrix(n, dtype=cfg.dtype)
    a = h + cfg.shift * np.eye(n, dtype=cfg.dtype)
    x_star = np.ones(n, dtype=cfg.dtype)
    b = a @ x_star # 一维数组自动适配矩阵
    return a, b, x_star

In [4]:
# 迭代求解器

# 辅助函数
def safe_l1(v: np.ndarray) -> float:
    """保证安全地计算 l1 范数。"""
    a = np.abs(v)
    if not np.all(np.isfinite(a)):
        return np.inf
    m = float(np.max(a))
    if m == 0.0:
        return 0.0
    # 缩放求和，降低溢出风险
    return m * float(np.sum(a / m))

def jacobi(
    a: np.ndarray, 
    b: np.ndarray, 
    x_star: np.ndarray, 
    cfg: IterConfig
) -> IterResult:
    """用 Jacobi 迭代出相应结果。"""
    # 初始化
    n = a.shape[0]
    x = np.zeros(n, dtype=cfg.dtype)
    d = np.diag(a) # A 的对角阵部分(此时为向量)

    # 如果 D 的逆不可求
    if np.any(np.isclose(d, 0.0)):
        return IterResult(n, "Jacobi", False, 0, np.inf, None)

    d_inv = 1.0 / d
    r = a - np.diag(d)

    err0 = safe_l1(x - x_star)

    if err0 < cfg.tol:
        return IterResult(n, "Jacobi", True, 0, err0, safe_l1(x))

    if cfg.max_iter <= 0:
        return IterResult(n, "Jacobi", False, 0, err0, None)

    # 限制，避免无意义计算
    blowup_limit = 1e100

    for k in range(1, cfg.max_iter + 1):
        try:
            # 将溢出和非法运算认为是异常。
            with np.errstate(over="raise", invalid="raise"):
                y = r @ x
                x_new = d_inv * (b - y)
        except FloatingPointError:
            # 一旦数值爆炸，则认为方法在第 k 步失败，并返回，不再继续迭代
            return IterResult(n, "Jacobi", False, k, np.inf, None)

        # 结果检查：检查是否 x_new 里有 inf 或 nan
        if not np.all(np.isfinite(x_new)):
            return IterResult(n, "Jacobi", False, k, np.inf, None)

        err = safe_l1(x_new - x_star)

        if (not np.isfinite(err)) or (err > blowup_limit):
            return IterResult(n, "Jacobi", False, k, np.inf, None)

        if err < cfg.tol:
            x_new_l1 = safe_l1(x_new)
            return IterResult(n, "Jacobi", True, k, err, x_new_l1)
        
        x = x_new

    return IterResult(n, "Jacobi", False, cfg.max_iter, err, None)

def gauss_seidel(
    a: np.ndarray, 
    b: np.ndarray, 
    x_star: np.ndarray, 
    cfg: IterConfig
) -> IterResult:
    """用 Gauss-Seidel 迭代法得出相应结果。"""
    n = a.shape[0]
    x = np.zeros(n, dtype=cfg.dtype)
    d = np.diag(a)
    err = np.inf

    if np.any(np.isclose(d, 0.0)):
        return IterResult(n, "Gauss-Seidel", False, 0, err, None)

    for k in range(1, cfg.max_iter + 1):
        x_old = x.copy()

        # 前代更新：x[i] 使用当前轮已更新的 x[:i] 和上一轮的 x_old[i+1:]
        for i in range(n):
            s1 = a[i, :i] @ x[:i]
            s2 = a[i, i + 1:] @ x_old[i + 1:]
            x[i] = (b[i] - s1 - s2) / a[i, i]

        err = safe_l1(x - x_star)
        if err < cfg.tol:
            x_l1 = safe_l1(x)
            return IterResult(n, "Gauss-Seidel", True, k, err, x_l1)

    return IterResult(n, "Gauss-Seidel", False, cfg.max_iter, err, None)

In [5]:
# 实验函数与结果呈现

def run_experiments(
    orders: list[int],
    cfg: IterConfig,
    solvers: list[Callable[[np.ndarray, np.ndarray, np.ndarray, IterConfig], IterResult]]
) -> list[IterResult]:
    """依据实验设置，批量运行，得到结果。"""
    results: list[IterResult] = []

    for n in orders:
        a, b, x_star = build_problem(n, cfg)
        for solver in solvers:
            results.append(solver(a, b, x_star, cfg))

    return results

def _fmt_float(v: Optional[float]) -> str:
    """格式化浮点数。"""
    if v is None:
        return "-"
    return f"{v:.6e}"

def print_results(results: list[IterResult]) -> None:
    """结果展示。"""
    header = f"{'n':>6}  {'method':>14}  {'conv':>6}  {'iters':>10}  {'||xk-x*||1':>14}  {'||xk||1':>14}"
    print(header)
    print("=" * len(header))
    for r in results:
        print(
            f"{r.n:6d}  {r.method:>14}  {str(r.converged):>6}  {r.iterations:10d}  "
            f"{_fmt_float(r.error_l1):>14}  {_fmt_float(r.x_l1):>14}"
        )

In [6]:
# 执行实验

cfg = IterConfig(
    tol=1e-5,
    max_iter=500000,
    shift=1.8,
    dtype=np.float64
)

solvers = [jacobi, gauss_seidel]
orders = [10, 20, 60, 100, 200, 500, 2000]

results = run_experiments(orders, cfg, solvers)
print_results(results)

     n          method    conv       iters      ||xk-x*||1         ||xk||1
    10          Jacobi    True          27    7.229326e-06    1.000001e+01
    10    Gauss-Seidel    True           7    3.937517e-06    9.999997e+00
    20          Jacobi    True          45    9.724454e-06    2.000001e+01
    20    Gauss-Seidel    True           8    6.413413e-06    2.000000e+01
    60          Jacobi    True         145    9.898077e-06    6.000001e+01
    60    Gauss-Seidel    True          10    2.862235e-06    6.000000e+01
   100          Jacobi    True         476    9.838068e-06    9.999999e+01
   100    Gauss-Seidel    True          10    9.268947e-06    1.000000e+02
   200          Jacobi   False        4781             inf               -
   200    Gauss-Seidel    True          11    7.268257e-06    2.000000e+02
   500          Jacobi   False        1729             inf               -
   500    Gauss-Seidel    True          12    8.768108e-06    5.000000e+02
  2000          Jacobi   

In [11]:
# 辅助探究函数

import numpy as np

def spectral_radius(G):
    """计算矩阵 G 的谱半径"""
    # 计算所有特征值
    eigenvalues = np.linalg.eigvals(G)
    # 返回特征值模的最大值
    return np.max(np.abs(eigenvalues))

def get_jacobi_radius(A):
    """计算系数矩阵 A 对应的 Jacobi 迭代阵谱半径"""
    D = np.diag(np.diag(A))
    Rest = A - D
    # B_J = -D^-1 * (L + U)
    inv_D = np.linalg.inv(D)
    B_J = -inv_D @ Rest

    return spectral_radius(B_J)

h = hilbert_matrix(200, dtype=np.float64)
a = h + 1.8 * np.eye(200, dtype=np.float64)
get_jacobi_radius(a)

np.float64(1.048291511614873)

In [10]:
for i in range(100, 201):
    h = hilbert_matrix(i, dtype=np.float64)
    a = h + 1.8 * np.eye(i, dtype=np.float64)
    if get_jacobi_radius(a) >= 1:
        print(f"在 {i} 阶时，Jacobi 已经发散。")
        break

在 131 阶时，Jacobi 已经发散。


### 2. 数值结果

下表汇总两种迭代方法在不同规模下的实验结果（保留 6 位小数）。

注：$x^* = (1, 1, ..., 1)^T$ 为精确解。 

| n | 迭代法 | 是否收敛 | 迭代步数 | 绝对误差 $\|x^{(k)}-x^*\|_1$ | $\|x^{(k)}\|_1$ |
|:--:|:--:|:--:|--:|--:|--:|
| 10 | Jacobi | True | 27 | 7.229326e-06 | 1.000001e+01 |
| 10 | Gauss-Seidel | True | 7 | 3.937517e-06 | 9.999997e+00 |
| 20 | Jacobi | True | 45 | 9.724454e-06 | 2.000001e+01 |
| 20 | Gauss-Seidel | True | 8 | 6.413413e-06 | 2.000000e+01 |
| 60 | Jacobi | True | 145 | 9.898077e-06 | 6.000001e+01 |
| 60 | Gauss-Seidel | True | 10 | 2.862235e-06 | 6.000000e+01 |
| 100 | Jacobi | True | 476 | 9.838068e-06 | 9.999999e+01 |
| 100 | Gauss-Seidel | True | 10 | 9.268947e-06 | 1.000000e+02 |
| 200 | Jacobi | False | 4781 | inf | - |
| 200 | Gauss-Seidel | True | 11 | 7.268257e-06 | 2.000000e+02 |
| 500 | Jacobi | False | 1729 | inf | - |
| 500 | Gauss-Seidel | True | 12 | 8.768108e-06 | 5.000000e+02 |
| 2000 | Jacobi | False | 1011 | inf | - |
| 2000 | Gauss-Seidel | True | 14 | 4.307426e-06 | 2.000000e+03 |

---

## 四. 算法分析

1.  **收敛判别准则**：

    *   迭代法 $x^{(k+1)} = Gx^{(k)} + f$ 收敛的充要条件是谱半径 $\rho(G) < 1$。
    *   **Jacobi 失败原因**：实验观测到 $n \ge 200$ 时 Jacobi 发散。通过特征值计算发现，随着 $n$ 增大，迭代阵 $G_J = -D^{-1}(L+U)$ 的谱半径逐渐超过 1（例如 $n=200$ 时 $\rho(G_J) \approx 1.05$），导致误差在迭代中被指数级放大，最终引发 `inf` 溢出。
    *   **Gauss-Seidel 的稳定性**：由于 Hilbert 矩阵 $H$ 是对称正定的，且 $1.8I$ 是正定对角阵，其和 $A = H + 1.8I$ 也是对称正定矩阵。由定理可知， GS 算法在所有测试规模下均能保持收敛。

2. **收敛速度**：

   *   GS 利用分量实时更新，收敛速度远超 Jacobi（如 $n=100$ 时快约 47 倍）。
   *   值得注意的是，在 $n=60, 100$ 等规模下，GS 的迭代步数基本保持在 10 步左右，表现出极强的病态抵抗能力。

3. **数值稳定性处理**：

   *   **异常拦截**：未加处理时，Jacobi 方法的发散会导致严重的浮点溢出（`Overflow`）与非法运算（`Invalid`）警告。
   *   **优化策略**：通过 `np.errstate` 实时捕获异常并结合 `safe_l1` 安全计算 1-范数，实现了在数值爆炸初期的安全退出，极大地避免了无意义计算。

---

## 五. 实验结论

1. **性能对比**：针对 Hilbert 偏移矩阵，**Gauss-Seidel 迭代全面优于 Jacobi**，在稳定性、收敛速度及适用规模上均具有压倒性优势。
2. **理论一致性**：实验验证了**对称正定性**是迭代法收敛的核心保障。尽管 Hilbert 矩阵具有极高的条件数，极易放大计算误差，但 Gauss-Seidel 凭借"对称正定矩阵必收敛"及分量实时更新的特性，在 $n = 2000$ 的大规模病态场景下依然保持了极高的稳健性。
3. **数值健壮性**：在迭代开发中引入**异常拦截机制**（如捕获溢出、安全计算范数）至关重要。该机制能有效规避数值爆炸引发的计算资源浪费。

**总结**：Gauss-Seidel 是求解本问题的稳健优选，而 Jacobi 不适用于大规模病态场景。

> 本次实验暂时无法体现 Jacobi 能并行运算的优越性，但在实际应用中，针对大规模稀疏矩阵，Jacobi 的并行化潜力仍不可忽视。 
